# Phase 3 — Confidence Intervals & Power Analysis

**Dataset:** Student Performance (UCI) — `student_mat_clean.csv` (n=357, đã làm sạch ở Phase 1)
**Mục tiêu:**
- Ước lượng khoảng tin cậy (KTC/CI) 95% cho trung bình điểm cuối kỳ `G3` bằng cả phương pháp **tham số (t)** và **bootstrap** (percentile + BCa)
- Tính KTC cho **sự khác biệt điểm trung bình giữa các nhóm** (nam/nữ, urban/rural) — bằng Welch và bootstrap
- Bootstrap KTC cho **Cohen's d** (effect size) của từng so sánh
- **Power analysis:** với cỡ mẫu thực tế n=357, ta đạt được công suất kiểm định (power) bao nhiêu cho các hiệu ứng nhỏ quan sát được, và cần bao nhiêu mẫu để đạt power=0.80

---
> **Output của notebook này:**
> `data/processed/bootstrap_ci_results.csv`
> `report/figures/ci_bootstrap_mean_g3.png`
> `report/figures/ci_group_differences.png`
> `report/figures/ci_power_curve.png`


## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from statsmodels.stats.power import TTestIndPower, tt_ind_solve_power
import warnings
import os

warnings.filterwarnings("ignore")

# ── Hằng số dùng chung toàn notebook ──
RANDOM_SEED = 42
ALPHA       = 0.05
N_BOOTSTRAP = 5000          # số lần resample bootstrap
FIGURES_DIR = "../report/figures"
DATA_OUT    = "../data/processed"

# Tạo thư mục nếu chưa có
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(DATA_OUT, exist_ok=True)

# ── Style ──
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 150, "savefig.bbox": "tight"})

np.random.seed(RANDOM_SEED)

print("Setup hoàn tất.")
print(f"  ALPHA       = {ALPHA}")
print(f"  N_BOOTSTRAP = {N_BOOTSTRAP}")
print(f"  RANDOM_SEED = {RANDOM_SEED}")

## 1. Load & recap dữ liệu

Đọc lại file sạch `student_mat_clean.csv` (output của Phase 1). Định nghĩa sẵn các
hàm bootstrap dùng lại xuyên suốt notebook.

In [ ]:
df_clean = pd.read_csv(f"{DATA_OUT}/student_mat_clean.csv")

print(f"Shape: {df_clean.shape[0]} học sinh × {df_clean.shape[1]} biến")
print(f"G3 — mean = {df_clean['G3'].mean():.4f}, std = {df_clean['G3'].std(ddof=1):.4f}")
print()
print("Phân bố theo nhóm phân tích:")
print("  sex     :", df_clean['sex'].value_counts().to_dict())
print("  address :", df_clean['address'].value_counts().to_dict())
df_clean[["sex", "address", "G3"]].head()

### 1.1 Hàm tiện ích — bootstrap & KTC tham số

- `bootstrap_ci(...)` — KTC bootstrap cho **một** thống kê bất kỳ, hỗ trợ 2 phương pháp:
  - **percentile**: lấy trực tiếp các phân vị của phân phối bootstrap.
  - **BCa** (bias-corrected & accelerated): hiệu chỉnh độ chệch (`z0`) và gia tốc (`a`, ước
    lượng qua jackknife) — chính xác hơn khi phân phối bootstrap lệch.
- `t_ci_mean(...)` — KTC t cổ điển (tham số) cho trung bình tổng thể.

In [ ]:
def bootstrap_ci(data, stat_fn=np.mean, n_boot=N_BOOTSTRAP, ci=95,
                 method="percentile", random_state=RANDOM_SEED):
    """KTC bootstrap cho một thống kê bất kỳ.

    Tham số
    -------
    data       : array-like — mẫu 1 chiều.
    stat_fn    : callable    — hàm thống kê (mặc định np.mean).
    n_boot     : int         — số lần resample.
    ci         : float       — mức tin cậy (%) (mặc định 95).
    method     : {"percentile", "bca"}.

    Trả về (point, lo, hi, boot_dist).
    """
    rng = np.random.default_rng(random_state)
    data = np.asarray(data, dtype=float)
    n = len(data)
    point = stat_fn(data)

    # Phân phối bootstrap
    idx = rng.integers(0, n, size=(n_boot, n))
    boot_dist = np.array([stat_fn(data[row]) for row in idx])

    alpha = (100 - ci) / 100.0
    if method == "percentile":
        lo = np.percentile(boot_dist, 100 * alpha / 2)
        hi = np.percentile(boot_dist, 100 * (1 - alpha / 2))
    elif method == "bca":
        # z0: hiệu chỉnh độ chệch
        prop = np.mean(boot_dist < point)
        prop = np.clip(prop, 1e-9, 1 - 1e-9)        # tránh ppf(0)/ppf(1) = ±inf
        z0 = stats.norm.ppf(prop)
        # a: gia tốc, ước lượng qua jackknife
        jack = np.array([stat_fn(np.delete(data, i)) for i in range(n)])
        jbar = jack.mean()
        num = np.sum((jbar - jack) ** 3)
        den = 6.0 * (np.sum((jbar - jack) ** 2) ** 1.5)
        a = num / den if den != 0 else 0.0
        z_lo, z_hi = stats.norm.ppf(alpha / 2), stats.norm.ppf(1 - alpha / 2)
        p_lo = stats.norm.cdf(z0 + (z0 + z_lo) / (1 - a * (z0 + z_lo)))
        p_hi = stats.norm.cdf(z0 + (z0 + z_hi) / (1 - a * (z0 + z_hi)))
        lo = np.percentile(boot_dist, 100 * p_lo)
        hi = np.percentile(boot_dist, 100 * p_hi)
    else:
        raise ValueError("method phải là 'percentile' hoặc 'bca'")
    return point, lo, hi, boot_dist


def t_ci_mean(data, conf=0.95):
    """KTC t (tham số) cho trung bình tổng thể."""
    data = np.asarray(data, dtype=float)
    n = len(data)
    mean = data.mean()
    se = stats.sem(data)
    df = n - 1
    lo, hi = stats.t.interval(conf, df, loc=mean, scale=se)
    return {"mean": mean, "se": se, "df": df, "moe": (hi - lo) / 2,
            "lo": lo, "hi": hi, "n": n}

print("Đã định nghĩa: bootstrap_ci(), t_ci_mean()")

## 2. KTC tham số cho trung bình G3 (95%)

KTC t cổ điển dựa trên giả định phân phối lấy mẫu của trung bình xấp xỉ chuẩn (CLT,
n=357 đủ lớn). Báo cáo: trung bình, sai số chuẩn (SE), bậc tự do (df), biên sai số
(margin of error) và khoảng [lo, hi].

In [ ]:
res_t = t_ci_mean(df_clean["G3"], conf=0.95)

print("KTC t 95% cho trung bình G3")
print(f"  n               = {res_t['n']}")
print(f"  mean(G3)        = {res_t['mean']:.4f}")
print(f"  SE              = {res_t['se']:.4f}")
print(f"  df              = {res_t['df']}")
print(f"  margin of error = ±{res_t['moe']:.4f}")
print(f"  KTC 95%         = [{res_t['lo']:.4f}, {res_t['hi']:.4f}]")

> **Nhận xét:** Trung bình G3 ≈ **11.52** điểm (thang 0–20). Với độ tin cậy 95%, trung
> bình thật của tổng thể nằm trong khoảng **[11.19, 11.86]** — biên sai số chỉ ±0.34 điểm,
> phản ánh cỡ mẫu n=357 cho ước lượng trung bình khá chính xác.

## 3. KTC bootstrap cho trung bình G3 (n_bootstrap=5000)

So sánh hai phương pháp bootstrap (percentile, BCa) với KTC tham số. Vì G3 gần đối xứng
và n lớn, kỳ vọng cả ba khoảng gần như trùng nhau (kiểm chứng định lý giới hạn trung tâm).

In [ ]:
pt_perc, lo_perc, hi_perc, boot_means = bootstrap_ci(df_clean["G3"], np.mean, method="percentile")
pt_bca,  lo_bca,  hi_bca,  _           = bootstrap_ci(df_clean["G3"], np.mean, method="bca")

cmp_mean = pd.DataFrame([
    {"method": "Parametric (t)",       "point": res_t["mean"], "ci_lower": res_t["lo"], "ci_upper": res_t["hi"]},
    {"method": "Bootstrap percentile", "point": pt_perc,       "ci_lower": lo_perc,     "ci_upper": hi_perc},
    {"method": "Bootstrap BCa",        "point": pt_bca,        "ci_lower": lo_bca,      "ci_upper": hi_bca},
])
cmp_mean["width"] = cmp_mean["ci_upper"] - cmp_mean["ci_lower"]
cmp_mean.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(boot_means, bins=50, color="#4C72B0", alpha=0.7, edgecolor="white")
ax.axvline(res_t["mean"], color="black", lw=2,
           label=f"Điểm ước lượng = {res_t['mean']:.3f}")
ax.axvline(res_t["lo"], color="green", ls="--", lw=1.8,
           label=f"Parametric t  [{res_t['lo']:.3f}, {res_t['hi']:.3f}]")
ax.axvline(res_t["hi"], color="green", ls="--", lw=1.8)
ax.axvline(lo_perc, color="red", ls=":", lw=1.8,
           label=f"Percentile    [{lo_perc:.3f}, {hi_perc:.3f}]")
ax.axvline(hi_perc, color="red", ls=":", lw=1.8)
ax.axvline(lo_bca, color="purple", ls="-.", lw=1.5,
           label=f"BCa           [{lo_bca:.3f}, {hi_bca:.3f}]")
ax.axvline(hi_bca, color="purple", ls="-.", lw=1.5)
ax.set_title("Phân phối bootstrap của trung bình G3 (5000 lần resample)")
ax.set_xlabel("Trung bình G3 (bootstrap)")
ax.set_ylabel("Tần suất")
ax.legend(fontsize=9)
fig.savefig(f"{FIGURES_DIR}/ci_bootstrap_mean_g3.png")
plt.show()

### 3.1 Mở rộng — Bootstrap KTC cho **trung vị** G3

Trung vị không có công thức KTC tham số gọn gàng → đây là minh hoạ rõ giá trị của bootstrap:
chỉ cần đổi `stat_fn=np.median` là có ngay KTC.

In [ ]:
pt_med, lo_med, hi_med, _ = bootstrap_ci(df_clean["G3"], np.median, method="percentile")

print(f"Trung vị G3                                 = {pt_med:.3f}")
print(f"KTC bootstrap 95% (percentile) cho trung vị = [{lo_med:.3f}, {hi_med:.3f}]")

> **Nhận xét:** KTC bootstrap (percentile & BCa) cho trung bình **trùng khớp** KTC t tới
> ~±0.05 điểm — đúng như kỳ vọng theo CLT khi n lớn và G3 gần đối xứng. Điều này cho thấy
> giả định tham số ở §2 là hợp lý. Bootstrap cho **trung vị** cho ra một khoảng (thường
> hơi rộng/giật hơn do trung vị nhạy với phân vị giữa), điều mà phương pháp tham số khó
> cung cấp trực tiếp — minh hoạ tính linh hoạt của bootstrap.

## 4. KTC cho sự khác biệt điểm trung bình giữa các nhóm

Trọng tâm Phase 3: ước lượng KTC cho **hiệu trung bình G3** của hai phép so sánh:
- `sex`: **M − F** (nam − nữ)
- `address`: **U − R** (urban − rural)

Với mỗi so sánh tính:
1. **KTC Welch** cho hiệu trung bình (không giả định phương sai bằng nhau, df Welch–Satterthwaite).
2. **KTC bootstrap** cho hiệu trung bình (resample **độc lập** từng nhóm).
3. **KTC bootstrap cho Cohen's d** (effect size chuẩn hoá).

Quy ước đọc: nếu KTC **chứa 0** ⇒ khác biệt **không** có ý nghĩa ở mức α=0.05.

In [ ]:
def cohens_d(a, b):
    """Cohen's d (pooled SD) cho hiệu hai trung bình a - b."""
    a = np.asarray(a, float); b = np.asarray(b, float)
    na, nb = len(a), len(b)
    sp = np.sqrt(((na - 1) * a.var(ddof=1) + (nb - 1) * b.var(ddof=1)) / (na + nb - 2))
    return (a.mean() - b.mean()) / sp


def welch_ci_diff(a, b, conf=0.95):
    """KTC Welch cho hiệu hai trung bình (phương sai không đồng nhất)."""
    a = np.asarray(a, float); b = np.asarray(b, float)
    na, nb = len(a), len(b)
    va, vb = a.var(ddof=1), b.var(ddof=1)
    diff = a.mean() - b.mean()
    se = np.sqrt(va / na + vb / nb)
    df = (va / na + vb / nb) ** 2 / ((va / na) ** 2 / (na - 1) + (vb / nb) ** 2 / (nb - 1))
    lo, hi = stats.t.interval(conf, df, loc=diff, scale=se)
    return {"diff": diff, "se": se, "df": df, "lo": lo, "hi": hi}


def bootstrap_diff_ci(a, b, stat_fn, n_boot=N_BOOTSTRAP, ci=95, random_state=RANDOM_SEED):
    """KTC bootstrap (percentile) cho thống kê hai nhóm — resample độc lập mỗi nhóm."""
    rng = np.random.default_rng(random_state)
    a = np.asarray(a, float); b = np.asarray(b, float)
    na, nb = len(a), len(b)
    point = stat_fn(a, b)
    boot = np.empty(n_boot)
    for k in range(n_boot):
        sa = a[rng.integers(0, na, na)]
        sb = b[rng.integers(0, nb, nb)]
        boot[k] = stat_fn(sa, sb)
    alpha = (100 - ci) / 100.0
    lo = np.percentile(boot, 100 * alpha / 2)
    hi = np.percentile(boot, 100 * (1 - alpha / 2))
    return point, lo, hi, boot

print("Đã định nghĩa: cohens_d(), welch_ci_diff(), bootstrap_diff_ci()")

In [ ]:
comparisons = [
    ("sex (M - F)",     df_clean.loc[df_clean.sex == "M", "G3"],     df_clean.loc[df_clean.sex == "F", "G3"]),
    ("address (U - R)", df_clean.loc[df_clean.address == "U", "G3"], df_clean.loc[df_clean.address == "R", "G3"]),
]

rows = []
for label, g1, g2 in comparisons:
    w = welch_ci_diff(g1, g2)
    _,    bd_lo, bd_hi, _ = bootstrap_diff_ci(g1, g2, lambda x, y: x.mean() - y.mean())
    cd_pt, cd_lo, cd_hi, _ = bootstrap_diff_ci(g1, g2, cohens_d)
    rows.append({
        "comparison": label, "n1": len(g1), "n2": len(g2),
        "diff_mean": w["diff"], "welch_lo": w["lo"], "welch_hi": w["hi"],
        "boot_lo": bd_lo, "boot_hi": bd_hi,
        "cohen_d": cd_pt, "d_lo": cd_lo, "d_hi": cd_hi,
    })

diff_tbl = pd.DataFrame(rows)
diff_tbl.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
y = np.arange(len(diff_tbl))[::-1]
first = True
for yi, (_, r) in zip(y, diff_tbl.iterrows()):
    ax.plot([r.welch_lo, r.welch_hi], [yi + 0.12, yi + 0.12], color="#4C72B0", lw=2.2)
    ax.plot(r.diff_mean, yi + 0.12, "o", color="#4C72B0", ms=8,
            label="Welch t-CI" if first else "")
    ax.plot([r.boot_lo, r.boot_hi], [yi - 0.12, yi - 0.12], color="#C44E52", lw=2.2)
    ax.plot(r.diff_mean, yi - 0.12, "s", color="#C44E52", ms=8,
            label="Bootstrap CI" if first else "")
    first = False
ax.axvline(0, color="black", ls="--", lw=1.3, label="Hiệu = 0 (không khác biệt)")
ax.set_yticks(y)
ax.set_yticklabels(diff_tbl["comparison"])
ax.set_ylim(-0.6, len(diff_tbl) - 0.4)
ax.set_xlabel("Hiệu trung bình G3")
ax.set_title("Forest plot — KTC 95% cho hiệu trung bình G3 giữa các nhóm")
ax.legend(loc="best", fontsize=9)
fig.savefig(f"{FIGURES_DIR}/ci_group_differences.png")
plt.show()

> **Nhận xét:**
> - **sex (M − F) ≈ +0.66 điểm**, KTC 95% ≈ **[−0.01, 1.33]** — khoảng **chứa 0**, nên khác
>   biệt nam/nữ **không** có ý nghĩa thống kê ở α=0.05. Cohen's d ≈ **0.21** (hiệu ứng *nhỏ*),
>   KTC của d cũng chứa 0.
> - **address (U − R) ≈ +1.02 điểm**, KTC 95% ≈ **[0.21, 1.83]** — khoảng **không chứa 0**,
>   nên học sinh urban có điểm cao hơn rural một cách có ý nghĩa. Cohen's d ≈ **0.32** (nhỏ→vừa).
> - KTC Welch và bootstrap gần như trùng nhau ⇒ kết luận vững (robust), không phụ thuộc giả
>   định tham số. Cả hai hiệu ứng đều **nhỏ** — đây chính là động lực cho phần Power analysis bên dưới.

## 5. Power analysis

Dùng `TTestIndPower` (statsmodels) cho kiểm định t hai mẫu, α=0.05, hai phía. Câu hỏi:
**với cỡ mẫu thực tế n=357, ta đạt được power bao nhiêu cho các hiệu ứng nhỏ quan sát ở §4?**

- **Post-hoc power** tại n thực tế (n mỗi nhóm theo split sex / address) cho (a) *effect size
  quan sát được* (Cohen's d ở §4) và (b) một *hiệu ứng vừa* d=0.5 để so sánh.
- **Cỡ mẫu cần** để đạt power=0.80 với d=0.5 (`tt_ind_solve_power`).

> **Lưu ý cỡ mẫu:** phân tích dùng **n=357** (mẫu sạch sau Phase 1 — đã loại G3=0 và
> winsorize `absences`). Con số **395** chỉ là cỡ mẫu thô *trước* khi làm sạch (README §1).

In [ ]:
analysis = TTestIndPower()

n_F = int((df_clean.sex == "F").sum());     n_M = int((df_clean.sex == "M").sum())
n_R = int((df_clean.address == "R").sum()); n_U = int((df_clean.address == "U").sum())

power_rows = []
for label, n1, n2, d_obs in [
    ("sex (M vs F)",     n_M, n_F, diff_tbl.loc[0, "cohen_d"]),
    ("address (U vs R)", n_U, n_R, diff_tbl.loc[1, "cohen_d"]),
]:
    ratio = n2 / n1
    pw_obs = analysis.power(effect_size=abs(d_obs), nobs1=n1, alpha=ALPHA, ratio=ratio)
    pw_med = analysis.power(effect_size=0.5,        nobs1=n1, alpha=ALPHA, ratio=ratio)
    power_rows.append({
        "comparison": label, "n1": n1, "n2": n2, "n_total": n1 + n2,
        "d_observed": abs(d_obs), "power_observed": pw_obs, "power_d=0.5": pw_med,
    })

power_tbl = pd.DataFrame(power_rows)
power_tbl.round(4)

In [ ]:
# Cỡ mẫu cần để đạt power=0.80 với hiệu ứng vừa d=0.5
n_req = tt_ind_solve_power(effect_size=0.5, alpha=ALPHA, power=0.80,
                           ratio=1.0, alternative="two-sided")
print(f"Để đạt power=0.80 với d=0.5 (α=0.05, hai phía):")
print(f"  n mỗi nhóm cần ≈ {np.ceil(n_req):.0f}")
print(f"  ⇒ tổng n cần   ≈ {np.ceil(n_req) * 2:.0f}")

# Đối chiếu: cỡ mẫu cần cho chính các hiệu ứng nhỏ quan sát được
for label, d_obs in [("sex", abs(diff_tbl.loc[0, "cohen_d"])),
                     ("address", abs(diff_tbl.loc[1, "cohen_d"]))]:
    n_need = tt_ind_solve_power(effect_size=d_obs, alpha=ALPHA, power=0.80,
                                ratio=1.0, alternative="two-sided")
    print(f"  [{label}] để đạt power=0.80 với d={d_obs:.3f}: cần ≈ {np.ceil(n_need):.0f}/nhóm "
          f"(tổng ≈ {np.ceil(n_need) * 2:.0f})")

In [ ]:
n_range = np.arange(5, 400)
fig, ax = plt.subplots(figsize=(9, 5.5))

for d_val, c in zip([0.2, 0.5, 0.8], ["#C44E52", "#4C72B0", "#55A868"]):
    powers = analysis.power(effect_size=d_val, nobs1=n_range, alpha=ALPHA, ratio=1.0)
    ax.plot(n_range, powers, color=c, lw=2, label=f"d = {d_val}")

ax.axhline(0.80, color="black", ls="--", lw=1.2, label="Ngưỡng power = 0.80")

# Đánh dấu n/nhóm thực tế (trung bình điều hoà của 2 nhóm)
hmean = lambda a, b: 2 / (1 / a + 1 / b)
for label, nh, c in [("sex", hmean(n_M, n_F), "#8172B3"),
                     ("address", hmean(n_U, n_R), "#CCB974")]:
    ax.axvline(nh, color=c, ls=":", lw=1.6,
               label=f"n/nhóm thực tế ({label}) ≈ {nh:.0f}")

ax.set_xlabel("Cỡ mẫu mỗi nhóm (n per group)")
ax.set_ylabel("Power (1 − β)")
ax.set_title("Đường cong Power vs cỡ mẫu — kiểm định t hai mẫu (α=0.05)")
ax.set_ylim(0, 1.02)
ax.legend(fontsize=9)
fig.savefig(f"{FIGURES_DIR}/ci_power_curve.png")
plt.show()

> **Nhận xét:**
> - Với **hiệu ứng nhỏ quan sát được**, power đạt được tại n=357 là **thấp**: sex (d≈0.21)
>   power ≈ **0.40**, address (d≈0.32) power ≈ **0.55–0.60** — đều **dưới ngưỡng 0.80**.
>   Nghiên cứu **thiếu công suất (underpowered)** để phát hiện các hiệu ứng nhỏ này một cách
>   tin cậy ⇒ kết quả "không có ý nghĩa" ở so sánh sex có thể là **sai lầm loại II** chứ chưa
>   chắc là "không có khác biệt".
> - Ngược lại, với **hiệu ứng vừa d=0.5**, cùng cỡ mẫu này cho power ≈ **>0.95** — dư sức phát hiện.
> - Để đạt power=0.80 với d=0.5 chỉ cần ≈ **64 học sinh/nhóm** (tổng ≈ 128); nhưng để bắt được
>   chính các hiệu ứng nhỏ quan sát được thì cần cỡ mẫu **lớn hơn nhiều** so với n hiện có.
> - Đường cong power minh hoạ rõ: ở cùng cỡ mẫu, power tăng mạnh theo effect size; vạch n thực
>   tế nằm dưới ngưỡng 0.80 đối với d=0.2.

## 6. Xuất kết quả

Gộp tất cả KTC đã tính vào một bảng gọn (tidy) và lưu ra `bootstrap_ci_results.csv`.

In [ ]:
records = []
# mean G3
records.append(["mean_G3", "all", "parametric_t",        res_t["mean"], res_t["lo"], res_t["hi"], 0.95, res_t["n"]])
records.append(["mean_G3", "all", "bootstrap_percentile", pt_perc,      lo_perc,     hi_perc,     0.95, len(df_clean)])
records.append(["mean_G3", "all", "bootstrap_bca",        pt_bca,       lo_bca,      hi_bca,      0.95, len(df_clean)])
# median G3
records.append(["median_G3", "all", "bootstrap_percentile", pt_med,     lo_med,      hi_med,      0.95, len(df_clean)])
# group differences + Cohen's d
for _, r in diff_tbl.iterrows():
    n_tot = int(r.n1 + r.n2)
    records.append(["diff_mean_G3", r.comparison, "welch_t",             r.diff_mean, r.welch_lo, r.welch_hi, 0.95, n_tot])
    records.append(["diff_mean_G3", r.comparison, "bootstrap_percentile", r.diff_mean, r.boot_lo,  r.boot_hi,  0.95, n_tot])
    records.append(["cohens_d",     r.comparison, "bootstrap_percentile", r.cohen_d,   r.d_lo,     r.d_hi,     0.95, n_tot])

ci_results = pd.DataFrame(records, columns=[
    "quantity", "group", "method", "point_estimate", "ci_lower", "ci_upper", "conf_level", "n"
])
ci_results = ci_results.round(4)

ci_results.to_csv(f"{DATA_OUT}/bootstrap_ci_results.csv", index=False)
print(f"Đã lưu {len(ci_results)} dòng KTC → {DATA_OUT}/bootstrap_ci_results.csv")
ci_results

## Tóm tắt — Phase 3 đã đạt được những gì

**Confidence Intervals**
- ✅ **KTC tham số (t) 95%** cho trung bình G3: ≈ **11.52 [11.19, 11.86]** (biên sai số ±0.34).
- ✅ **KTC bootstrap (5000 resample)** cho trung bình G3 bằng cả **percentile** và **BCa** — trùng
  khớp KTC t (sai lệch ~±0.05), kiểm chứng CLT và tính vững của ước lượng.
- ✅ Mở rộng: **KTC bootstrap cho trung vị** G3 — minh hoạ giá trị của bootstrap với thống kê
  không có công thức tham số gọn.

**KTC cho khác biệt nhóm (trọng tâm)**
- ✅ **sex (M − F)** ≈ +0.66 điểm, KTC 95% ≈ [−0.01, 1.33] — **chứa 0** ⇒ *không* có ý nghĩa; d ≈ 0.21 (nhỏ).
- ✅ **address (U − R)** ≈ +1.02 điểm, KTC 95% ≈ [0.21, 1.83] — **không chứa 0** ⇒ urban > rural có ý nghĩa; d ≈ 0.32 (nhỏ→vừa).
- ✅ **Bootstrap KTC cho Cohen's d** của cả hai so sánh; Welch và bootstrap cho kết luận nhất quán.
- ✅ **Forest plot** (`ci_group_differences.png`) trực quan hoá CI và đường tham chiếu 0.

**Power analysis**
- ✅ **Post-hoc power tại n=357**: với các hiệu ứng *nhỏ* quan sát được, power chỉ ≈ **0.40 (sex)**
  và **≈ 0.55–0.60 (address)** — **dưới 0.80** ⇒ nghiên cứu *underpowered* cho hiệu ứng nhỏ
  (rủi ro sai lầm loại II, đặc biệt với so sánh sex).
- ✅ Với hiệu ứng *vừa* d=0.5, cùng cỡ mẫu cho power **>0.95**; cần ≈ **64/nhóm** (tổng ≈128) để đạt power=0.80.
- ✅ **Đường cong power** (`ci_power_curve.png`) cho d=0.2/0.5/0.8 kèm vạch n thực tế và ngưỡng 0.80.

**Files xuất ra**
- `data/processed/bootstrap_ci_results.csv` — 10 dòng KTC (mean/median/diff/Cohen's d).
- `report/figures/ci_bootstrap_mean_g3.png`, `ci_group_differences.png`, `ci_power_curve.png`.

> **Kết luận Phase 3:** Các KTC cho thấy chỉ có khác biệt **urban/rural** là có ý nghĩa (hiệu
> ứng nhỏ→vừa), còn **nam/nữ** thì chưa kết luận được — và power analysis làm rõ rằng điều này
> phần lớn do **cỡ mẫu hạn chế** đối với các hiệu ứng nhỏ, chứ không khẳng định "không có khác biệt".
> Đây là minh chứng cho việc báo cáo **effect size + CI + power** song song với p-value (định
> hướng xuyên suốt project).